In [1]:
# Differentiable symplectic integrator in jax

In [2]:
# Optimization points:
# -- usage of jit/vmap and their ordering
# -- perfomance/memory optimization for one initial condition
# -- optimize vmap over initials
# -- optimize nested derivatives

In [3]:
# Import

from typing import Any
from typing import Sequence
from typing import Callable
from typing import Optional

from functools import reduce
from itertools import groupby

import jax
from jax import Array
from jax import jit
from jax import vmap
from jax import grad

In [4]:
# Utilities

def nest(length: int, mapping: Callable[..., Array]) -> Callable[..., Array]:
    def closure(x: Array, *args: Any) -> Array:
        def scan_body(carry: tuple[Array, tuple], _: Any) -> tuple[tuple[Array, tuple], None]:
            x, args = carry
            x = mapping(x, *args)
            return (x, args), None
        (x, *_), _ = jax.lax.scan(scan_body, (x, args), None, length=length)
        return x
    return closure


def fold(mappings: Sequence[Callable[..., Array]]) -> Callable[..., Array]:
    idxs: Array = jax.numpy.arange(len(mappings))
    def closure(x: Array, *args: Any) -> Array:
        def scan_body(carry: tuple[Array, tuple], idx: Array) -> tuple[tuple[Array, tuple], None]:
            x, args = carry
            x = jax.lax.switch(idx, mappings, x, *args)
            return (x, args), None
        (x, *_), _ = jax.lax.scan(scan_body, (x, args), idxs)
        return x
    return closure

In [5]:
# Yoshida composition construction functions


def first(xs:Iterable[Any]) -> Any:
    x, *_ = xs
    return x


def last(xs:Iterable[Any]) -> Any:
    *_, x = xs
    return x


def weights(n:int) -> list[float]:
    return [
        +1/(2 - 2**(1/(1 + 2*n))),
        -2**(1/(1 + 2*n))/(2 - 2**(1/(1 + 2*n))),
        +1/(2 - 2**(1/(1 + 2*n)))
    ]


def coefficients(ni:int,  nf:int) -> list[float]:
    ws = map(weights, range(ni if ni != 0 else 1, nf + 1))
    return reduce(lambda xs, x: [xi*xsi for xi in x for xsi in xs], ws, [1.0])


def table(k:int, ni:int, nf:int, merge:bool=False) -> tuple[list[int], list[float]]:
    ps = [[i, 0.5] for i in range(k - 1)]
    ps = ps + [[k - 1, 1.0]] + [*reversed(ps)]
    ns, vs = map(list, zip(*ps))
    cs = coefficients(ni, nf)
    ps = sum(([[n, v*c] for (n, v) in zip(ns, vs)] for c in cs), start = [])
    if merge:
        gs = groupby(ps, key=first)
        ps = [reduce(lambda x, y: [first(x), last(x) + last(y)], g) for _, g in gs]
    return tuple([*map(list, zip(*ps))])


def sequence(ni:int,
             nf:int,
             mappings:Sequence[Callable[..., Array]],
             merge:bool=False,
             parameters:Optional[list[list[Any]]]=None) ->  Sequence[Callable[..., Array]]:
    indices, weights, *_ = table(len(mappings), ni, nf, merge)
    parameters = [[] for _ in range(len(mappings))] if parameters is None else parameters
    parameters = [parameters[i] for i in indices]
    def wrapper(mapping, weight, parameter):
        def function(x, dt, *args):
            return mapping(x, weight*dt, *args, *parameter)
        return function
    return [
        wrapper(mappings[index], weight, parameter)
        for index, weight, parameter in zip(indices, weights, parameters)
    ]

In [6]:
# Implicit integrator

def midpoint(H:Callable[..., Array],
            ns:int=1,
            gradient:Optional[Callable[..., Array]] = None,
            jacobian:Optional[Callable[..., Array]] = None,
            solve:Optional[Callable[[Array, Array], Array]] = None) -> Callable[..., Array]:
    gradient = grad if gradient is None else gradient
    jacobian = jax.jacrev if jacobian is None else jacobian
    if solve is None:
        def solve(matrix:Array, vector:Array) -> Array:
            return jax.numpy.linalg.solve(matrix, vector)    
    dHdq = gradient(H, argnums=0)
    dHdp = gradient(H, argnums=1)    
    def integrator(state: Array, dt: Array, t: Array, *args: Array) -> Array:
        q, p = jax.numpy.reshape(state, (2, -1))
        t_m = t + 0.5*dt
        def residual(state: Array) -> tuple[Array, Array]:
            Q, P = jax.numpy.reshape(state, (2, -1))
            q_m = 0.5*(q + Q)
            p_m = 0.5*(p + P)
            dq = Q - q - dt*dHdp(q_m, p_m, t_m, *args)
            dp = P - p + dt*dHdq(q_m, p_m, t_m, *args)
            state = jax.numpy.concatenate([dq, dp])
            return state, state
        auxiliary = jacobian(residual, has_aux=True)
        def newton(state: Array) -> Array:
            matrix, error = auxiliary(state)
            return state + solve(matrix, -error)
        return nest(ns, newton)(state)
    return integrator

In [7]:
# Set data type

jax.config.update("jax_enable_x64", True)

In [8]:
# Set device

device, *_ = jax.devices('cpu')
jax.config.update('jax_default_device', device)

In [9]:
# Construct Yoshida step (multi-map integrator)

# H = H1 + H2
# H1 = 1/2 q**2 + 1/3 q**3 -> [q, p] -> [q, p - t*q - t*q**2]
# H2 = 1/2 p**2            -> [q, p] -> [q + t*q, p]

# Set mappings for sovable parts

def fn(x, t):
    q, p = x
    return jax.numpy.stack([q, p - t*(q + q**2)])

def gn(x, t):
    q, p = x
    return jax.numpy.stack([q + t*p, p])

# Generate Yoshida sequence

fs = sequence(0, 2, [fn, gn], merge=True)

# Generate folded step (sequence composition)

integrator = fold(fs)

# Set parameters

dt = jax.numpy.array(0.01)
x = jax.numpy.array([0.1, -0.05])

# Compile several integration steps

step = jit(nest(10, integrator))
xa = step(x, dt)
xa

Array([ 0.09446052, -0.06067929], dtype=float64)

In [10]:
# Construct Yoshida composition step (implicit midpoint)

# Define hamiltonian function

def h(q, p, t, *args):
    return jax.numpy.sum(1/2*(q**2 + p**2) + 1/3*q**3)

# Define implicit step

integrator = midpoint(h, ns=1)

# Generate Yoshida sequence

fs = sequence(0, 2, [integrator], merge=False)

# Generate folded step (sequence composition)

integrator = fold(fs)

# Set parameters

dt = jax.numpy.array(0.01)
t = jax.numpy.array(0.0)
x = jax.numpy.array([0.1, -0.05])

# Compile several steps

step = jit(nest(10, integrator))
xb = step(x, dt, t)
xb

Array([ 0.09446052, -0.06067929], dtype=float64)

In [11]:
print(xa - xb)

[-1.63757896e-15 -2.77555756e-15]


In [12]:
xs = jax.numpy.stack(64*[x])
xs.shape

(64, 2)

In [13]:
vmap(step, (0, None, None))(xs, dt, t).shape

(64, 2)